# 异质性分析-产权性质  
将AED因子数据分割为过期和民企
分别进行组合构建，分桶，FM回归  

## 导入库

In [57]:
import os
import dotenv
import re
import warnings
import polars as pl
import plotly  
import plotly.express as px
from plotly.subplots import make_subplots
import statsmodels.api as sm
dotenv.load_dotenv()

True

## 超参数

In [58]:
# 基本配置
BASELINE_TASK_ID_PREFIX = 'baseline1'  # 基线任务id前缀
SAVE_BASE_DIR = f'/home/frank/files/programs/GraduationThesis/empirical/{BASELINE_TASK_ID_PREFIX}' # 保存基本路
SAVE = True # 是否保存数据

# 数据库
CONNECTION_URL = os.getenv("POSTGRES_URL")
ENGINE = "adbc"

# 异质性组
EQU = 1 # 1为国企，0为其他


## 读取数据
读取MA因子数据和市值数据，划分为大市值组和小市值组

In [59]:
ma_df = pl.read_parquet(SAVE_BASE_DIR + '/MA因子_copy.parquet')
ma_df.head()

date,portfolio,MA,return
date,str,f64,f64
2023-01-01,"""301019""",0.673006,0.0132
2022-11-01,"""002010""",0.811182,-0.069629
2017-03-01,"""600770""",0.607574,-0.1574
2024-03-01,"""300512""",0.839616,-0.0327
2006-11-01,"""002031""",0.649186,0.0552


In [60]:
en_equality_nature = pl.read_database_uri(
    uri=CONNECTION_URL,
    query='''SELECT DISTINCT symbol AS "portfolio", equitynature
    FROM com_info.en_equitynatureall''',
    engine=ENGINE,
)

In [61]:
en_equality_nature.head()

portfolio,equitynature
str,str
"""600888""","""民营"""
"""002510""","""民营"""
"""600886""","""国企"""
"""300972""","""民营"""
"""600236""","""国企"""


## 划分数据
生成 EQU 

In [62]:
en_equality_nature = en_equality_nature.drop_nulls('equitynature')
en_equality_nature = en_equality_nature.with_columns(
    pl.when(pl.col('equitynature') == '国企').then(1).otherwise(0).alias('equity')
).drop('equitynature')

和ma_df合并，保存为 MA因子.parquet 文件

In [63]:
ma_df = ma_df.join(en_equality_nature,on=['portfolio'],how='left')
ma_df = ma_df.filter(pl.col('equity') == EQU).drop_nulls('equity').drop('equity')

## 保存数据

In [64]:
ma_df.write_parquet(SAVE_BASE_DIR + '/MA因子.parquet')

**保存后，运行2，3，4 notebook，查看异质性结果** 